In [9]:
import pandas as pd
import numpy as np
import re
import itertools
from itertools import chain, combinations, product

In [10]:
#  Descr: get domain of the generators
#  Input: specific generator list (list with lists)
# Output: list with all symptoms (domain)
def getdom(gens):
    domain_list = []
    for gen in gens:
        if isinstance(gen[0], list):
            domain_list = domain_list + [e for s in gen[0] for e in s] + [e for s in gen[1] for e in s]
        else:
            domain_list = domain_list + [e for s in gen[:-1] for e in s]
    return domain_list

#  Descr: single integer to binary combination based on a specific domain
#  Input: single integer, domain (list of symptoms), bin = as binary or symptom list
# Output: symptom set or binary representation
def int2comb(integer, domain, bin=True):
    temp = format(integer,'0'+str(len(domain))+'b')
    if bin:
        return temp
    else:
        return set([domain[idx] for idx in [i for i,s in enumerate(temp) if s=='1']])

#  Descr: integer list to binary combination based on a specific domain
#  Input: integer list, domain (list of symptoms), bin = as binary or symptom list
# Output: list of symptom sets or binary representation list
def int2comb_list(integers, domain, bin=True):
    temp = [format(i,'0'+str(len(domain))+'b') for i in integers]
    if bin:
        return temp
    else:
        return [set([domain[idx] for idx in [i for i,s in enumerate(t) if s=='1']]) for t in temp]

In [11]:
#  Descr: reads a binary file and converts the data back to list of symptom combinations (as an integer list)
#  Input: file path
# Output: numpy array in unsigned integer 64 format
def read_all_comb(file_path):
    chunk = 5_000 #chunk size to read
    chunk_np = 1_000_000 #chunk size to write to numpy array
    chunk_list = []
    chunk_np_list = []
    combis = np.empty([0,1],dtype=np.uint64)
    counter = 0
    
    with open(file_path,"rb") as f:
        bytes2read = f.read(1) # first byte is always the domain length
        bytes2read = (int.from_bytes(bytes2read, byteorder='big')+7) // 8
        #print("Domain Length: ", int.from_bytes(bytes2read, byteorder='big'))
        #print("Bytes to Read: ", bytes2read)
        
        while True:
            entry = f.read(bytes2read)
            if not entry:
                break
            chunk_list.append(entry)
            
            counter = counter + 1
            if counter >= chunk:
                chunk_np_list += [np.uint64(int.from_bytes(a, byteorder='big')) for a in chunk_list]
                chunk_list = []
                counter = 0
                
            if len(chunk_np_list) >= chunk_np:
                combis = np.append(combis, np.array(chunk_np_list))
                chunk_np_list = []
                
        if chunk_list:
            chunk_np_list += [np.uint64(int.from_bytes(a, byteorder='big')) for a in chunk_list]
            combis = np.append(combis, np.array(chunk_np_list))
    
    return combis

In [12]:
def recommendNext(df_list, has_symptoms):
    df_all_disorders = pd.concat(df_list)
    df_all_disorders = df_all_disorders.fillna(0)
    df_filtered = df_all_disorders
    
    # Non-Absent Symptoms Variant: string_query = ' and '.join([s+'==1' for s in has_symptoms])
    # Differentiate between present (s+'==1') and absent symptoms (s+'==0'), indicated by ! before name
    # "symptom1", "!symptom2"
    query_parts = []
    clean_symptoms = []
    
    for s in has_symptoms:
        if s.startswith('!'):
            clean_name = s[1:]  # Removes the starting "!"
            query_parts.append(f"{clean_name}==0")
            clean_symptoms.append(clean_name)
        else:
            query_parts.append(f"{s}==1")
            clean_symptoms.append(s)    
    string_query = ' and '.join(query_parts)
    
    df_filtered = df_filtered.query(string_query)
    df_filtered = df_filtered.drop(clean_symptoms, axis=1)
    disorders_coulds = set(df_filtered['disorder'].tolist())
    
    print("Possible disorders:",", ".join(disorders_coulds))
    print()
    
    df_grouped  = df_filtered.groupby('disorder', as_index=False).agg('mean')
    df_grouped['1'] = df_grouped.apply(lambda row: row[row == 1].index.tolist(), axis=1)
    df_grouped['0'] = df_grouped.apply(lambda row: row[row == 0].index.tolist(), axis=1)
    
    #alternative: set(itertools.chain.from_iterable(df_grouped['1'].tolist()))
    diff1 = set(sum(df_grouped['1'].tolist(),[]))
    diff0 = set(sum(df_grouped['0'].tolist(),[]))
    symptoms2check = diff0 & diff1
    
    print("Important symptoms to check:",", ".join(symptoms2check))
    print()
    
    for s in symptoms2check:
        disorders0 = set()
        disorders1 = set()
        for index, row in df_grouped.iterrows():
            if s in row['1']:
                disorders1.add(row['disorder'])
            if s in row['0']:
                disorders0.add(row['disorder'])
        print("Check", s, "to differentiate between", ", ".join(disorders1), "and", ", ".join(disorders0))

In [13]:
def dec2bin(number):
    binary_str = bin(number)[2:]
    return [int(digit) for digit in binary_str]

def array2dataframe(names_list, array):
    bin_lists = [dec2bin(num) for num in array]
    max_len = max(len(bin_list) for bin_list in bin_lists)
    padded_lists = [([0] * (max_len - len(bin_list)) + bin_list) for bin_list in bin_lists]

    df = pd.DataFrame(padded_lists)
    df.columns = names_list[1:]
    df.insert(loc=0, column='disorder', value=names_list[0])
    
    return df

def getdom4test(lists):
    dom_list = []
    for l in lists:
        for e in l:
            if e not in dom_list:
                dom_list.append(e)
    return dom_list

def getdomainFromFile(name):
    domain = []
    with open('disorder_domains.txt', 'r') as f:
        for line in f:
            disorder = line.rstrip().split(',')
            if disorder[0]==name:
                domain = disorder[1:]
    return domain

### Disorders

In [14]:
d1 = [[1,1,1,0,1,1,1,1,0,1],
      [1,1,0,0,1,1,1,1,1,0],
      [1,0,0,0,1,1,1,1,1,0],
      [0,0,0,0,1,1,1,0,0,0],
      [0,0,0,0,1,1,0,1,0,0]]
d1 = [['symptom1','symptom2','symptom3','symptom5','symptom6','symptom7','symptom8','symptom10'],
      ['symptom1','symptom2','symptom5','symptom6','symptom7','symptom8','symptom9'],
      ['symptom1','symptom5','symptom6','symptom7','symptom8','symptom9'],
      ['symptom5','symptom6','symptom7'],
      ['symptom5','symptom6','symptom8']]
      
d2 = [[0,0,1,1,1,1,1,1,0,1],
      [0,0,0,0,0,1,1,1,0,0],
      [0,0,1,0,1,1,1,1,0,0]]
d2 = [['symptom3','symptom4','symptom5','symptom6','symptom7','symptom8','symptom10'],
      ['symptom6','symptom7','symptom8'],
      ['symptom3','symptom5','symptom6','symptom7','symptom8']]

d3 = [[1,0,0,1,1,1,1,1,1,1],
      [0,0,0,0,1,1,1,0,0,0]]
d3 = [['symptom1','symptom4','symptom5','symptom6','symptom7','symptom8','symptom9','symptom10'],
      ['symptom5','symptom6','symptom7']]

d4 = [[1,0,0,0,1,0,0,1,0,1]]
d4 = [['symptom1','symptom5','symptom8','symptom10']]

In [15]:
all_dis_names = ['disorder1','disorder2','disorder3','disorder4']
all_dis_lists = []

for dis_name in all_dis_names:
    dis1_arr = read_all_comb(dis_name+".bin")
    dis1_names = getdomainFromFile(dis_name)
    dis1_names.insert(0, dis_name)
    all_dis_lists.append(array2dataframe(dis1_names, dis1_arr))   

In [16]:
has_symptoms = ['symptom'+str(i) for i in [5,6,7,8]]
has_symptoms
recommendNext(all_dis_lists, has_symptoms)

Possible disorders: disorder3, disorder1, disorder2

Important symptoms to check: symptom3, symptom4, symptom9, symptom1

Check symptom3 to differentiate between disorder2 and disorder3
Check symptom4 to differentiate between disorder3 and disorder1
Check symptom9 to differentiate between disorder3 and disorder2
Check symptom1 to differentiate between disorder3, disorder1 and disorder2
